# 11 · Running experiments

Guide 10 tuned models with the train pipeline. This guide is about running such
experiments for real, when they take hours and something gets in the way:

- naming a run, and the run-state manifest WarpRec keeps for it;
- pausing a train run with a signal and resuming it, and what a changed
  configuration does to a resume;
- what a finished run leaves on disk;
- dashboards: CodeCarbon, MLflow and Weights & Biases;
- the estimate pipeline, which sizes an experiment before it runs;
- the swarm pipeline, which tunes every model at once.

The train and swarm runs go through the command line, `python -m warprec.run
-c <config> -p <pipeline>`, started from the notebook as a subprocess: a pause
is a signal sent to a process, and that is what this guide sends.

**Requirements:** `pip install "warprec[dashboard]" jupyter` (the `dashboard`
extra brings CodeCarbon, MLflow and W&B). Runs in 10 to 20 minutes on a
laptop CPU, depending on its load; each run starts and stops its own local Ray
instance.

In [1]:
import json
import re
import shutil
import signal
import subprocess
import sys
import time
from pathlib import Path

import pandas as pd

sys.path.insert(0, "..")
from guide_data import movielens_100k

movielens_100k()
RUNS = Path("../runs/11-running-experiments")
shutil.rmtree(RUNS, ignore_errors=True)  # a re-run must not resume the last one
RUNS.mkdir(parents=True)
pd.set_option("display.width", 160)
pd.set_option("display.max_colwidth", 100)

`movielens_100k()` downloads MovieLens-100K and writes `ratings.tsv`, as guide 1
showed. Two helpers: one starts a pipeline in a subprocess with its log going to
a file, the other prints the WarpRec lines of that log that contain one of the
given phrases (Ray's own output and the per-epoch tables are left out).

In [2]:
ANSI = re.compile(r"\x1b\[[0-9;]*m")


def start(config: str, pipeline: str, log: Path) -> subprocess.Popen:
    """Start `python -m warprec.run` in the background, logging to a file."""
    return subprocess.Popen(
        [sys.executable, "-m", "warprec.run", "-c", config, "-p", pipeline],
        stdout=log.open("w"),
        stderr=subprocess.STDOUT,
    )


def show_log(log: Path, *phrases: str) -> None:
    """Print the driver's WarpRec log lines containing any of the phrases."""
    for line in ANSI.sub("", log.read_text()).splitlines():
        if line.startswith("(") or " - WarpRec - " not in line:
            continue  # lines from Ray workers, or not from WarpRec
        message = line.split(" - WarpRec - ", 1)[1].split(" ", 1)[1]
        if any(p in message for p in phrases):
            print(message)

## Naming a run

The configuration below tunes EASE (three values of `l2`) and BPR (a 2×2 grid,
twelve epochs each) on a temporal train/validation/test split. What is new is the
`run` section: `run.name` gives the experiment an identity that survives the
process, and `run.resume` says what to do when a run of that name already
exists.

In [3]:
print(Path("configs/baselines.yml").read_text())

# Two models, each with a small grid: EASE (closed form, three trials) and BPR
# (iterative, four trials of twelve epochs). The run is named, so that it can be
# paused and resumed. Paths are relative to the guide's folder.
run:
  name: ml100k-baselines
  resume: auto
reader:
  loading_strategy: dataset
  data_type: transaction
  reading_method: local
  local_path: ../data/ml-100k/ratings.tsv
  rating_type: implicit
splitter:
  test_splitting:
    strategy: temporal_holdout
    ratio: 0.1
  validation_splitting:
    strategy: temporal_holdout
    ratio: 0.1
writer:
  dataset_name: baselines
  writing_method: local
  local_experiment_path: ../runs/11-running-experiments/
models:
  EASE:
    optimization:
      num_workers: 0
    l2: [grid, 50.0, 100.0, 200.0]
  BPR:
    optimization:
      num_workers: 0
      properties:
        seed: 42
    embedding_size: [grid, 16, 32]
    learning_rate: [grid, 0.001, 0.005]
    reg_weight: 0.0001
    batch_size: 1024
    epochs: 12
evaluation:
  to

The name is optional. Without it, WarpRec derives one from the writer's
`dataset_name` and the first 8 hex digits of a fingerprint of the configuration;
`resolve_run_name` is the function the pipelines call. `configs/dashboards.yml`,
used later, sets no name:

In [4]:
from loguru import logger

from warprec.common import resolve_run_name
from warprec.utils.config import load_train_configuration

logger.disable("warprec")
print(resolve_run_name(load_train_configuration("configs/baselines.yml")))
print(resolve_run_name(load_train_configuration("configs/dashboards.yml")))

ml100k-baselines
dashboards_62408aac


A derived name changes whenever the data, split, evaluation or set of models
changes, so a run without a name can be resumed only with exactly the same
setup. An explicit name is what lets you find the run again.

## Pausing a run

The train pipeline turns `SIGINT` (Ctrl+C) and `SIGTERM` into a **pause**: it
asks Ray Tune to stop and checkpoint its experiment, records how far every model
got, and exits with status 0. A second signal aborts without saving.

The cell below starts the run, waits until BPR's sweep, the second model, has
reported at least two epochs, and sends `SIGINT`. Ray Tune writes each trial's
reports to `ray_results/<run>__<model>/<trial>/progress.csv`, which is what it
polls. (Graceful pausing is on by default; `run.pause_on_signal: false` turns it
off.)

In [5]:
OUT = RUNS / "baselines"
BPR_TRIALS = OUT / "ray_results" / "ml100k-baselines__BPR"


def epochs_per_trial() -> dict:
    """Number of epochs each BPR trial has reported so far."""
    epochs = {
        p.parent.name.split("_")[-1]: len(p.read_text().splitlines()) - 1
        for p in BPR_TRIALS.glob("*/progress.csv")
    }
    return dict(sorted(epochs.items()))


first_log = RUNS / "baselines-1.log"
started = time.time()
process = start("configs/baselines.yml", "train", first_log)
while process.poll() is None and sum(epochs_per_trial().values()) < 2:
    time.sleep(0.2)
process.send_signal(signal.SIGINT)
at_signal = epochs_per_trial()
print("exit status:", process.wait())
at_pause = epochs_per_trial()

exit status: 0


In [6]:
show_log(
    first_log,
    "Run name",
    "HPO for",
    "Pause requested",
    "interrupted",
    "paused",
    "Resume it",
)

Run name: ml100k-baselines
HPO for EASE ended successfully.
Pause requested. WarpRec will stop at the next safe point and save the run state. Interrupt again to abort immediately.
Hyperparameter optimization for BPR was interrupted. No best model will be selected from this partial sweep.
Run 'ml100k-baselines' has been paused. Progress is saved at ../runs/11-running-experiments/baselines/run_state/ml100k-baselines.json.
Resume it by running the same command again with run.name set to 'ml100k-baselines' and run.resume set to 'auto' or 'force'.


EASE finished; BPR's sweep was cut short. WarpRec does **not** pick a best BPR
from the trials that happened to have reported: it marks the sweep interrupted
and stops.

### The run-state manifest

What the run achieved is recorded in `<experiment_path>/run_state/<run>.json`.
At the top: the pipeline, the WarpRec version, the fingerprint of the run-level
configuration and the timestamp pinned into the output file names, so that a
resumed run keeps writing to the same `Overall_Results_*`, `Overall_Params_*` and
`Time_Report_*` files.

In [7]:
manifest_path = OUT / "run_state" / "ml100k-baselines.json"
manifest = json.loads(manifest_path.read_text())
{k: v for k, v in manifest.items() if k != "models"}

{'schema_version': 1,
 'run_name': 'ml100k-baselines',
 'pipeline': 'train',
 'warprec_version': '2.1.1',
 'created_at': '2026-10-08T12:33:36',
 'updated_at': '2026-10-08T12:35:29',
 'writer_timestamp': '08.10.2026_12-33-36',
 'config_fingerprint': 'c80465b1156159b0aa8881a2976a76026ee0c52e0bfbb4b1b4ad8acefef3671e'}

Then one entry per model, with its status and the fingerprint of its own
configuration. The status is one of `pending`, `interrupted`, `hpo_completed`
(sweep done, evaluation or writing not), `completed` and `failed`. A model
whose sweep finished also keeps its best parameters, best iteration and best
checkpoint, so a pause during evaluation does not cost the sweep.

In [8]:
def model_table(manifest: dict) -> pd.DataFrame:
    return pd.DataFrame(
        {
            name: {
                "status": state["status"],
                "fingerprint": state["fingerprint"][:12] + "…",
                "tune experiment": state["tune_experiment_name"],
                "best params": state["best_params"],
                "best iter": state["best_iter"],
            }
            for name, state in manifest["models"].items()
        }
    ).T


model_table(manifest)

,status,fingerprint,tune experiment,best params,best iter
EASE,completed,e027b30314d3…,ml100k-baselines__EASE,{'l2': 100.0},1
BPR,interrupted,33bc8bee06e1…,ml100k-baselines__BPR,None,0


## Resuming

Resuming is running exactly the same command again. With `resume: auto`, WarpRec
finds the manifest, skips the completed model and hands BPR's Ray Tune
experiment back to Ray Tune, which continues each unfinished trial from its last
checkpoint (weights, optimizer, scheduler, epoch counter, early-stopping state).

In [9]:
second_log = RUNS / "baselines-2.log"
process = start("configs/baselines.yml", "train", second_log)
print("exit status:", process.wait())
print(f"both runs: {time.time() - started:.0f} s")
show_log(second_log, "Resuming run", "Skipping", "resumable", "HPO for", "concluded")

exit status: 0
both runs: 223 s
Resuming run 'ml100k-baselines' created at 2026-10-08T12:33:36.
Skipping EASE: already completed in this run.
Found a resumable Ray Tune experiment at guides/11-running-experiments/../runs/11-running-experiments/baselines/ray_results/ml100k-baselines__BPR. Unfinished trials will continue from their last checkpoint.
HPO for BPR ended successfully.
All experiments concluded. WarpRec is shutting down.


The trials did not start over: the epochs they had reported before the pause
are still in their `progress.csv`, and the resumed session appended the rest
(a trial at 0 had not started when the signal came).

In [10]:
pd.DataFrame(
    {
        "at the signal": at_signal,
        "when the run stopped": at_pause,
        "after the resume": epochs_per_trial(),
    }
).fillna(0).astype(int).sort_index().rename_axis("BPR trial")

,at the signal,when the run stopped,after the resume
BPR trial,,,
00000,3,3,12
00001,3,3,12
00002,3,3,12
00003,3,3,12


In [11]:
model_table(json.loads(manifest_path.read_text()))

,status,fingerprint,tune experiment,best params,best iter
EASE,completed,e027b30314d3…,ml100k-baselines__EASE,{'l2': 100.0},1
BPR,completed,33bc8bee06e1…,ml100k-baselines__BPR,"{'embedding_size': 16, 'reg_weight': 0.0001, 'batch_size': 1024, 'epochs': 12, 'learning_rate': ...",11


## What a finished run leaves

Guide 10 walked through these files; this is the layout under
`<local_experiment_path>/<dataset_name>/`:

In [12]:
for path in sorted(OUT.iterdir()):
    if path.is_dir():
        children = sorted(p.name for p in path.iterdir())
        shown = [c for c in children if not c.startswith("ray_train_trial_id")]
        hidden = len(children) - len(shown)
        print(
            f"{path.name}/",
            shown + ([f"+{hidden} trial checkpoint folders"] if hidden else []),
        )
    else:
        print(path.name)

evaluation/ ['Overall_Results_08.10.2026_12-33-36.tsv', 'Time_Report_08.10.2026_12-33-36.tsv']
params/ ['Overall_Params_08.10.2026_12-33-36.json']
ray_results/ ['ml100k-baselines__BPR', 'ml100k-baselines__EASE', '+7 trial checkpoint folders']
recs/ []
run_state/ ['ml100k-baselines.json']
serialized/ []
split/ []


`evaluation/` holds the test results and the time report, `params/` the best
hyperparameters, `run_state/` the manifest, and `ray_results/` one Ray Tune
experiment per model plus the trial checkpoints. `recs/`, `serialized/` and
`split/` fill when `meta.save_recs`, `meta.save_model` and `writer.save_split`
ask for them. Both sessions wrote to the files of the first one:

In [13]:
pd.read_csv(next((OUT / "evaluation").glob("Overall_Results_*.tsv")), sep="\t")

,Model,Top@k,Recall,nDCG
0,BPR,10,0.091076,0.099345
1,EASE,10,0.106698,0.117068


The time report (`general.time_report`, on by default) has one row per model:
exploration, evaluation and inference time, parameter counts and the RAM the
trials used.

In [14]:
time_report = pd.read_csv(
    next((OUT / "evaluation").glob("Time_Report_*.tsv")), sep="\t"
)
time_report.set_index("Model Name").T

Model Name,BPR,EASE
Trainable Params (Best Model),40928,0
Total Params (Best Model),40928,0
Data Preparation Time,0:00:13.743376,0:00:16.468619
Hyperparameter Exploration Time,0:01:03.793869,0:00:40.355517
RAM Mean Usage (MB),361.269531,455.484375
RAM Max Usage (MB),497.390625,462.5625
Average Trial Time (s),51.417208,26.824641
Evaluation Time,0:00:00.133739,0:00:00.179892
Total Time,0:01:17.670984,0:00:57.004028
Inference Time (ms),41.373968,73.925257


## What invalidates a resume

| `run.resume` | an existing, compatible manifest | no manifest, or an incompatible one |
|---|---|---|
| `auto` (default) | resume | start a fresh run, with a warning |
| `force` | resume | raise an error |
| `never` | discard it, with a warning, and start a fresh run | start a fresh run |

Compatibility is decided by two fingerprints. The run fingerprint covers
`reader`, `filtering`, `splitter`, `evaluation` and the set of model names; a
change there invalidates the whole run. A model's fingerprint covers its own
block, minus the resource settings (`cpu_per_trial`, `gpu_per_trial`,
`custom_resources_per_trial`, `label_selector`, `max_concurrent_trials`,
`num_workers`, `device`), so a paused run can be resumed on a machine of another
size. The `dashboard`, `general` and `run` sections are not fingerprinted.

In [15]:
from warprec.common import model_fingerprint

bpr = load_train_configuration("configs/baselines.yml").models["BPR"]
more_cpus = {**bpr, "optimization": {**bpr["optimization"], "cpu_per_trial": 4}}
fewer_epochs = {**bpr, "epochs": 3}
pd.Series(
    {
        "as configured": model_fingerprint("BPR", bpr)[:12],
        "cpu_per_trial: 4": model_fingerprint("BPR", more_cpus)[:12],
        "epochs: 3": model_fingerprint("BPR", fewer_epochs)[:12],
    },
    name="BPR fingerprint",
)

as configured       33bc8bee06e1
cpu_per_trial: 4    33bc8bee06e1
epochs: 3           daf9bfaa8fd7
Name: BPR fingerprint, dtype: object

In a scheduled job prefer `resume: force`: with `auto`, a typo in the name
silently starts the experiment from zero. `run.errored_trials` (`skip`,
`resume`, `restart`) chooses what happens to trials that errored before a
pause.

### A changed model

The run above is complete. Running it again with BPR at four epochs instead of
twelve changes BPR's fingerprint only: EASE is skipped, BPR is optimised again.
Its old Ray Tune experiment is not restored; it is moved aside to
`<run>__BPR.discarded-<timestamp>`, so nothing is deleted.

In [16]:
baselines = Path("configs/baselines.yml").read_text()
four_epochs = RUNS / "baselines-bpr-4-epochs.yml"
four_epochs.write_text(baselines.replace("epochs: 12", "epochs: 4"))

third_log = RUNS / "baselines-3.log"
process = start(str(four_epochs), "train", third_log)
print("exit status:", process.wait())
show_log(third_log, "Skipping", "changed", "was moved to", "HPO for")

exit status: 0
Skipping EASE: already completed in this run.
The configuration of BPR changed since the last run. Its saved state will be discarded and the model will be optimized from scratch.
The Ray Tune experiment at guides/11-running-experiments/../runs/11-running-experiments/baselines/ray_results/ml100k-baselines__BPR belongs to run state that was discarded. It was moved to guides/11-running-experiments/../runs/11-running-experiments/baselines/ray_results/ml100k-baselines__BPR.discarded-20261008-123726 and the model will be optimized from scratch.
HPO for BPR ended successfully.


In [17]:
def experiments() -> list:
    """The Ray Tune experiment folders of the run, current and set aside."""
    return sorted(
        re.sub(r"discarded-[\d-]+", "discarded-<timestamp>", p.name)
        for p in (OUT / "ray_results").iterdir()
        if p.name.startswith("ml100k-baselines__")
    )


print(experiments())
print("epochs per BPR trial:", epochs_per_trial())
model_table(json.loads(manifest_path.read_text()))

['ml100k-baselines__BPR', 'ml100k-baselines__BPR.discarded-<timestamp>', 'ml100k-baselines__EASE']
epochs per BPR trial: {'00000': 4, '00001': 4, '00002': 4, '00003': 4}


,status,fingerprint,tune experiment,best params,best iter
EASE,completed,e027b30314d3…,ml100k-baselines__EASE,{'l2': 100.0},1
BPR,completed,961ec8b7b387…,ml100k-baselines__BPR,"{'embedding_size': 16, 'reg_weight': 0.0001, 'batch_size': 1024, 'epochs': 4, 'learning_rate': 0...",4


The new BPR trials ran four epochs each, and the manifest holds BPR's new
fingerprint and best parameters.

### `resume: never`

`never` discards the whole manifest: every model is optimised again and every
old Ray Tune experiment is set aside. A changed `reader`, `filtering`,
`splitter` or `evaluation` section does the same under `auto`, after a warning
naming the mismatch. The configuration is the four-epoch one, with
`resume: never`:

In [18]:
never = RUNS / "baselines-never.yml"
never.write_text(four_epochs.read_text().replace("resume: auto", "resume: never"))

fourth_log = RUNS / "baselines-4.log"
process = start(str(never), "train", fourth_log)
print("exit status:", process.wait())
show_log(fourth_log, "set to 'never'", "was moved to", "HPO for")
print(experiments())

exit status: 0
Run state for 'ml100k-baselines' exists but resume is set to 'never'. It will be discarded and the run will start from scratch.
The Ray Tune experiment at guides/11-running-experiments/../runs/11-running-experiments/baselines/ray_results/ml100k-baselines__EASE belongs to run state that was discarded. It was moved to guides/11-running-experiments/../runs/11-running-experiments/baselines/ray_results/ml100k-baselines__EASE.discarded-20261008-123843 and the model will be optimized from scratch.
HPO for EASE ended successfully.
The Ray Tune experiment at guides/11-running-experiments/../runs/11-running-experiments/baselines/ray_results/ml100k-baselines__BPR belongs to run state that was discarded. It was moved to guides/11-running-experiments/../runs/11-running-experiments/baselines/ray_results/ml100k-baselines__BPR.discarded-20261008-123916 and the model will be optimized from scratch.
HPO for BPR ended successfully.
['ml100k-baselines__BPR', 'ml100k-baselines__BPR.discarded

Both models trained from scratch, and the new run keeps a manifest of its own,
so a later `auto` resumes it. It also writes its results under a new
timestamp:

In [19]:
sorted(p.name for p in (OUT / "evaluation").glob("Overall_Results_*.tsv"))

['Overall_Results_08.10.2026_12-33-36.tsv',
 'Overall_Results_08.10.2026_12-38-35.tsv']

## Dashboards

The `dashboard` section attaches Ray Tune logger callbacks to every sweep, in
addition to the JSON and CSV files Ray Tune always writes. Three are available,
all off by default, all from the `dashboard` extra:

- `codecarbon`: one CodeCarbon tracker per trial, from trial start to trial end;
- `mlflow`: every epoch's report and the trial's hyperparameters, to an MLflow
  tracking server or a local store;
- `wandb`: the same to Weights & Biases (needs an account and an API key).

WarpRec does not configure TensorBoard itself. The configuration below trains a
single BPR for three epochs with CodeCarbon and MLflow on, MLflow writing to a
local file store; the W&B block is shown commented out.

In [20]:
print(Path("configs/dashboards.yml").read_text())

# One BPR configuration trained for three epochs, with CodeCarbon writing the
# energy each trial used to a CSV and MLflow logging every epoch's report to a
# local file store. No run.name: WarpRec derives one.
reader:
  loading_strategy: dataset
  data_type: transaction
  reading_method: local
  local_path: ../data/ml-100k/ratings.tsv
  rating_type: implicit
splitter:
  test_splitting:
    strategy: temporal_holdout
    ratio: 0.1
  validation_splitting:
    strategy: temporal_holdout
    ratio: 0.1
writer:
  dataset_name: dashboards
  writing_method: local
  local_experiment_path: ../runs/11-running-experiments/
models:
  BPR:
    optimization:
      num_workers: 0
      properties:
        seed: 42
    embedding_size: 32
    learning_rate: 0.005
    reg_weight: 0.0001
    batch_size: 1024
    epochs: 3
evaluation:
  top_k: [10]
  metrics: [nDCG, Recall]
  validation_metric: nDCG@10
dashboard:
  codecarbon:
    enabled: true
    save_to_file: true
    tracking_mode: machine
  mlflow:

In [21]:
dashboards_log = RUNS / "dashboards.log"
process = start("configs/dashboards.yml", "train", dashboards_log)
print("exit status:", process.wait())

exit status: 0


CodeCarbon's `emissions.csv` lands in the run's `ray_results/` folder, one row
per trial, its `project_name` being the trial id. The trackers run in the Ray
Tune driver, not in the worker that trains, so keep the default
`tracking_mode: machine` (the whole machine): `process` measures the driver. On a
laptop without power counters CodeCarbon estimates CPU power from the
processor's TDP, so read the energy as an order of magnitude. (The file also
records the location CodeCarbon looked up for the carbon intensity; it is left
out here.)

In [22]:
emissions = pd.read_csv(RUNS / "dashboards" / "ray_results" / "emissions.csv")
emissions[
    [
        "project_name",
        "tracking_mode",
        "duration",
        "cpu_power",
        "ram_power",
        "energy_consumed",
        "emissions",
    ]
].T

,0
project_name,aa4ba_00000
tracking_mode,machine
duration,23.293921
cpu_power,10.399304
ram_power,6.0
energy_consumed,0.000101
emissions,0.000033


MLflow gets one run per trial in the experiment named in the configuration,
with the trial's hyperparameters as parameters. WarpRec names a metric
`<metric>@<k>`, which MLflow does not accept, so the dashboard logs it as
`<metric>/<k>`: `nDCG@10` arrives as `nDCG/10`. The store is read back here with
MLflow's own client, through its `file://` URI:

In [23]:
import mlflow

mlflow.set_tracking_uri((RUNS / "mlruns").resolve().as_uri())
runs = mlflow.search_runs(experiment_names=["warprec-guide-11"])
print(
    len(runs),
    "run;",
    "metrics:",
    sorted(c[8:] for c in runs if c.startswith("metrics.")),
)
history = mlflow.MlflowClient().get_metric_history(runs.run_id[0], "nDCG/10")
pd.DataFrame({"step": [m.step for m in history], "nDCG/10": [m.value for m in history]})

1 run; metrics: ['best_nDCG/10', 'completed_epochs', 'config/batch_size', 'config/embedding_size', 'config/epochs', 'config/learning_rate', 'config/reg_weight', 'done', 'epoch', 'iterations_since_restore', 'loss', 'nDCG/10', 'pid', 'ram_peak_mb', 'step', 'time_since_restore', 'time_this_iter_s', 'time_total_s', 'timestamp', 'train_loss', 'training_iteration']


,step,nDCG/10
0,1,0.074735
1,2,0.084871
2,3,0.094097


For a remote server, `tracking_uri` is its URL and `tracking_token` its token;
`registry_uri`, `tags` and `save_artifacts` are the other MLflow keys. For W&B,
`project`, `group` and `team` (the W&B entity) place the runs; `api_key_file`
or `api_key` authenticate; `excludes` drops keys from what is logged,
`log_config` also logs the trial's configuration and `upload_checkpoints` its
checkpoints. The dashboards are not part of the fingerprints, so they can be
switched on for a resumed run.

## Estimating before running

The estimate pipeline sizes an experiment without running it. It expands every
grid point of every model, builds the model, and:

- for an iterative model, times `train_batches` training batches and
  `eval_batches` evaluation batches after `warmup_batches` unmeasured ones, then
  extrapolates to an epoch, to all `epochs`, and to a full evaluation;
- for a closed-form model, asks the model for an analytical estimate of the
  memory its training needs (for EASE, the item × item matrices) and times
  nothing.

It runs in the calling process, without Ray, and only grid (or `choice`) search
spaces are accepted. The configuration sizes the same two models as
`baselines.yml`, under the same full evaluation; the evaluation `batch_size` is
lowered so that MovieLens' 943 users make enough batches to time after the
warm-up.

In [24]:
print(Path("configs/estimate.yml").read_text())

# The two models of baselines.yml, sized before running them: the estimate
# pipeline builds every grid point, times a few batches and extrapolates.
# Continuous search spaces are not allowed here, only grid/choice.
reader:
  loading_strategy: dataset
  data_type: transaction
  reading_method: local
  local_path: ../data/ml-100k/ratings.tsv
  rating_type: implicit
splitter:
  test_splitting:
    strategy: temporal_holdout
    ratio: 0.1
writer:
  dataset_name: estimate
  writing_method: local
  local_experiment_path: ../runs/11-running-experiments/
models:
  EASE:
    l2: [grid, 50.0, 100.0, 200.0]
  BPR:
    embedding_size: [grid, 16, 32]
    learning_rate: [grid, 0.001, 0.005]
    reg_weight: 0.0001
    batch_size: 1024
    epochs: 12
evaluation:
  top_k: [10]
  metrics: [nDCG, Recall]
  batch_size: 128
estimate:
  warmup_batches: 2
  train_batches: 20
  eval_batches: 5
general:
  device: cpu



`estimate_pipeline(path)` is what `-p estimate` runs. Its log prints two summary
lines per model; PyTorch Lightning's start-up messages are silenced.

In [25]:
import logging
import warnings

import ray

from warprec.pipelines import estimate_pipeline

logging.getLogger("lightning").setLevel(logging.ERROR)
logging.getLogger("lightning.pytorch").setLevel(logging.ERROR)
logger.enable("warprec")
logger.remove()
handler = logger.add(
    sys.stdout,
    format="{message}",
    filter=lambda r: any(
        k in r["message"] for k in ("Estimate summary", "RAM max", "failed")
    ),
)
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    estimate_pipeline("configs/estimate.yml")
logger.remove(handler)
logger.disable("warprec")
print("Ray started:", ray.is_initialized())

Estimate summary for EASE: setups=3, est_train=N/A, est_eval=N/A


Train RAM max=93.243 MB, Eval RAM max=N/A, Train VRAM max=N/A, Eval VRAM max=N/A


Estimate summary for BPR: setups=4, est_train=1.621 s, est_eval=0.015 s


Train RAM max=18.484 MB, Eval RAM max=22.785 MB, Train VRAM max=N/A, Eval VRAM max=N/A


Ray started: False


The report, `evaluation/Estimate_Report_<timestamp>.tsv`, has one row per model,
**averaged over its setups** (`Setup Count`): the mean of the setups' maxima,
not the largest setup. RAM is the growth of the estimating process's resident
memory over its level before the stage, sampled at every batch; the VRAM
columns fill only on CUDA.

In [26]:
estimate = pd.read_csv(
    next((RUNS / "estimate" / "evaluation").glob("Estimate_Report_*.tsv")), sep="\t"
)
report = estimate.set_index("Model Name").dropna(axis=1, how="all")
print(report.pop("Notes").to_string(), end="\n\n")
report.T

Model Name
BPR                                                                            CPU only
EASE    CPU only; Analytical train-space estimate; EASE analytical train-space estimate



Model Name,BPR,EASE
Setup Count,4,3
Measured Train Batches,20,0
Measured Eval Batches,5,0
Train Batch Time Avg,0.001535 s,NaN
Train Batch Time Std,0.000464 s,NaN
Estimated Train Epoch Time,0.135090 s,NaN
Estimated Total Train Time,1.621076 s,NaN
Eval Batch Time Avg,0.001886 s,NaN
Eval Batch Time Std,0.000396 s,NaN
Estimated Evaluation Time,0.015090 s,NaN


EASE has only the analytical training RAM (its note says so); BPR has measured
batch times, the epoch and total training time they extrapolate to, the
evaluation time and the memory samples. Compare its *Estimated Total Train
Time* with the per-trial times of the real run in the time report above: the
estimate covers the training loop only, not Ray's start-up for each trial nor
the per-epoch validation.

## The swarm pipeline

The train pipeline tunes models one after the other; inside a model's sweep the
trials run in parallel. The swarm pipeline (`-p swarm`, `swarm_pipeline(path)`)
launches every model's sweep at once as a Ray task, which fills a large cluster
better. It reads the same configuration as the train pipeline, writes the same
outputs, and its log interleaves the models. Its pause is coarser: each sweep
runs inside a Ray task, so the driver cancels the tasks and Ray Tune falls back
to its last periodic checkpoint. For a run you expect to pause, prefer the
train pipeline.

In [27]:
print(Path("configs/swarm.yml").read_text())

# The swarm pipeline takes the same configuration as the train pipeline. Two
# single-configuration models, tuned at the same time instead of one after the
# other. Both ask for their model to be saved, EASE also for its
# recommendations, and a Wilcoxon test compares the two.
run:
  name: ml100k-swarm
reader:
  loading_strategy: dataset
  data_type: transaction
  reading_method: local
  local_path: ../data/ml-100k/ratings.tsv
  rating_type: implicit
splitter:
  test_splitting:
    strategy: temporal_holdout
    ratio: 0.1
  validation_splitting:
    strategy: temporal_holdout
    ratio: 0.1
writer:
  dataset_name: swarm
  writing_method: local
  local_experiment_path: ../runs/11-running-experiments/
models:
  EASE:
    meta:
      save_model: true
      save_recs: true
    optimization:
      num_workers: 0
    l2: 100.0
  BPR:
    meta:
      save_model: true
    optimization:
      num_workers: 0
      properties:
        seed: 42
    embedding_size: 16
    learning_rate: 0.005
    

In [28]:
swarm_log = RUNS / "swarm.log"
process = start("configs/swarm.yml", "swarm", swarm_log)
print("exit status:", process.wait())
show_log(swarm_log, "Launching", "Swarming completed", "Statistical", "concluded")

exit status: 0
Launching the swarm of experiments. Number of different models: 2
Swarming completed. Aggregating and saving results.
Statistical significance test results written to ../runs/11-running-experiments/swarm/evaluation/Wilcoxon_test_08.10.2026_12-42-03.tsv
Statistical significance tests completed successfully.
Experiment swarming concluded. WarpRec is shutting down.


In [29]:
sorted(
    str(p.relative_to(RUNS / "swarm"))
    for p in (RUNS / "swarm").rglob("*")
    if p.is_file() and "ray_results" not in p.parts
)

['evaluation/Overall_Results_08.10.2026_12-42-03.tsv',
 'evaluation/Time_Report_08.10.2026_12-42-03.tsv',
 'evaluation/Wilcoxon_test_08.10.2026_12-42-03.tsv',
 'params/Overall_Params_08.10.2026_12-42-03.json',
 'recs/EASE_08.10.2026_12-42-03.tsv',
 'run_state/ml100k-swarm.json',
 'run_state/ml100k-swarm/eval/BPR.pt',
 'run_state/ml100k-swarm/eval/EASE.pt',
 'serialized/BPR_PrettyClassicScallop.pth',
 'serialized/EASE_PinkTrogonOfMastery.pth']

The same files as a train run: results, parameters, the time report, EASE's
recommendations, both saved models, and the Wilcoxon test over the two models
tuned side by side (*Accepted* marks a significant difference; guide 12 covers
the tests):

In [30]:
from IPython.display import display

evaluation = RUNS / "swarm" / "evaluation"
display(pd.read_csv(next(evaluation.glob("Overall_Results_*.tsv")), sep="\t"))
pd.read_csv(next(evaluation.glob("Wilcoxon_*.tsv")), sep="\t")

,Model,Top@k,Recall,nDCG
0,BPR,10,0.071373,0.084022
1,EASE,10,0.106698,0.117068


,Model A,Model B,Metric,Cutoff,Statistic,p-value,Significance (α=0.05)
0,EASE,BPR,Recall,10,31538.5,4.339474e-10,Accepted
1,EASE,BPR,nDCG,10,52303.5,2.177390e-10,Accepted


## Summary

| key | what it does |
|---|---|
| `run.name` | the run's identity: names the manifest `run_state/<name>.json` and the Ray Tune experiments `<name>__<model>`; derived from `dataset_name` and a config fingerprint when omitted |
| `run.resume` | `auto` resumes a compatible run or starts afresh, `force` resumes or fails, `never` discards the manifest and sets the old Ray Tune experiments aside |
| `run.errored_trials` | `skip`, `resume` or `restart` the trials that errored before a pause |
| `run.pause_on_signal` | turn SIGINT/SIGTERM into a graceful pause (default `true`) |
| `general.time_report` | write `Time_Report_*.tsv` (default `true`) |
| `dashboard.codecarbon` | `enabled`, `save_to_file`, `save_to_api`, `tracking_mode` (`machine` or `process`) |
| `dashboard.mlflow` | `enabled`, `tracking_uri`, `registry_uri`, `experiment_name`, `tags`, `tracking_token`, `save_artifacts` |
| `dashboard.wandb` | `enabled`, `project`, `group`, `team`, `api_key_file`/`api_key`, `excludes`, `log_config`, `upload_checkpoints` |
| `estimate` | `warmup_batches`, `train_batches`, `eval_batches` of the estimate pipeline |

The docs pages: [Pause & Resume](https://warprec.readthedocs.io/en/latest/pipelines/pause-resume/),
[Run configuration](https://warprec.readthedocs.io/en/latest/configuration/run/),
[Dashboard configuration](https://warprec.readthedocs.io/en/latest/configuration/dashboard/),
[Estimate pipeline](https://warprec.readthedocs.io/en/latest/pipelines/estimate/),
[Estimate configuration](https://warprec.readthedocs.io/en/latest/configuration/estimate/)
and [Swarm pipeline](https://warprec.readthedocs.io/en/latest/pipelines/swarm/).